# Local Green, Global Red III: 02 Why Zero Can Mislead
## What does a zero defect actually establish?

**Author:** Jeremy H. Carroll  
**Date:** 2026-10-15  
**Version:** 1.5.2  

---

### 1. Question and Prediction

> **Question:** If a test suite reports an aggregate defect sum of zero ($\sum_i \Delta_i = 0$), does that guarantee that no composition failures occurred?

**Prediction:** Before running the code below: can opposite errors cancel each other out in a sum?

The answer is **yes**: a positive defect (composite passing while a component failed) and a negative defect (LGGR failure) sum to zero, hiding both failures from any aggregate metric.

In [ ]:
from io import BytesIO
from IPython.display import Image, display
import matplotlib.pyplot as plt

def display_figure(fig):
    try:
        with BytesIO() as buffer:
            fig.savefig(buffer, format='png', dpi=160, bbox_inches='tight')
            display(Image(data=buffer.getvalue()))
    finally:
        plt.close(fig)

# Pointwise Defect Records
def compute_sample_summary(records: list[tuple[int, int, int]]):
    """Compute signed defects, retaining underlying pointwise records."""
    pointwise = [eg - (e1 * e2) for e1, e2, eg in records]
    n = len(pointwise)
    neg_count = sum(1 for d in pointwise if d < 0)
    zero_count = sum(1 for d in pointwise if d == 0)
    pos_count = sum(1 for d in pointwise if d > 0)
    signed_sum = sum(pointwise)
    abs_magnitude = sum(abs(d) for d in pointwise)
    return {
        'count': n,
        'pointwise': pointwise,
        'neg_count': neg_count,
        'zero_count': zero_count,
        'pos_count': pos_count,
        'signed_sum': signed_sum,
        'abs_magnitude': abs_magnitude,
    }

# Concrete Cancelling Example:
# Trial 1: (1, 1, 0) -> LGGR failure (Delta = -1)
# Trial 2: (0, 1, 1) -> Global-only pass (Delta = +1)
cancelling_sample = [(1, 1, 0), (0, 1, 1)]
s = compute_sample_summary(cancelling_sample)

print('Pointwise Records:')
for idx, (e1, e2, eg) in enumerate(cancelling_sample, 1):
    print(f'  Observation {idx}: (e1={e1}, e2={e2}, eg={eg}) -> Delta_{idx} = {eg - e1*e2}')
print('\nSummary Statistics:')
print(f'  Number of observations:    {s["count"]}')
print(f'  Negative defects (-1):     {s["neg_count"]}')
print(f'  Zero defects (0):          {s["zero_count"]}')
print(f'  Positive defects (+1):     {s["pos_count"]}')
print(f'  Total signed defect sum:   {s["signed_sum"]}  <-- ZERO!')
print(f'  Absolute magnitude sum:    {s["abs_magnitude"]} (descriptive summary)')

### 2. Visualizing Cancellation vs. Agreement

Notice how the signed sum completely masks the two severe errors.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))

# Left: Pointwise values
ax1.bar(['Obs 1: (1,1,0)', 'Obs 2: (0,1,1)'], [-1, 1], color=['#c44536', '#2f6690'], width=0.4)
ax1.axhline(0, color='#333333', linewidth=0.8)
ax1.set_ylim(-1.5, 1.5)
ax1.set_title('Pointwise Defects: Disagreements Present', fontsize=10)
ax1.set_ylabel('Signed Defect Delta_i', fontsize=10)
ax1.grid(axis='y', linestyle='--', alpha=0.4)

# Right: Aggregate sum
ax2.bar(['Aggregate Sum'], [0], color=['#888888'], width=0.3)
ax2.scatter(0, 0, color='#888888', s=100, zorder=3)
ax2.axhline(0, color='#333333', linewidth=0.8)
ax2.set_ylim(-1.5, 1.5)
ax2.set_title('Aggregate Sum: False Sense of Perfection', fontsize=10)
ax2.set_ylabel('Total Defect sum(Delta_i)', fontsize=10)
ax2.grid(axis='y', linestyle='--', alpha=0.4)

fig.tight_layout()
display_figure(fig)

### 3. Presets Explorer: Four Critical Regimes (Optional Widget)

Inspect how the summary behaves across four canonical presets:
1. **Pointwise Agreement:** All observations have $\Delta_i = 0$.
2. **Cancellation:** Non-zero errors of opposing sign cancel to sum $0$.
3. **Empty Sample:** No observations recorded ($N = 0$).
4. **Constant Evaluators:** Trivial components yielding constant outputs.

In [ ]:
presets = {
    'Pointwise Agreement': [(1, 1, 1), (1, 1, 1), (0, 0, 0)],
    'Cancellation': [(1, 1, 0), (0, 1, 1), (1, 1, 0), (1, 0, 1)],
    'Empty Sample': [],
    'Constant Evaluators': [(0, 0, 0), (0, 0, 0), (0, 0, 0)],
}

def render_preset(preset_name: str):
    records = presets[preset_name]
    s = compute_sample_summary(records)
    print(f'Preset: {preset_name}')
    print(f'  Underlying records:        {records}')
    print(f'  Pointwise defect list:     {s["pointwise"]}')
    print(f'  Observation count (N):     {s["count"]}')
    print(f'  Negative / Zero / Positive: {s["neg_count"]} / {s["zero_count"]} / {s["pos_count"]}')
    print(f'  Total signed defect sum:   {s["signed_sum"]}')
    print(f'  Absolute magnitude sum:    {s["abs_magnitude"]}')
    if s["count"] == 0:
        print('  Note: Empty sample has sum=0, but provides zero verification evidence.')
    elif s["signed_sum"] == 0 and s["abs_magnitude"] > 0:
        print('  Note: Cancellation detected! Signed sum masks severe individual errors.')
    print('-' * 60)

# Run all presets statically
print('--- Static Execution of All Four Presets ---')
for p in presets:
    render_preset(p)

# Optional interactive dropdown
import os
if os.environ.get('LGGR3_HEADLESS') != '1':
    try:
        import ipywidgets as widgets
        from IPython.display import display
        drop = widgets.Dropdown(options=list(presets.keys()), value='Cancellation', description='Preset:')
        out = widgets.interactive_output(render_preset, {'preset_name': drop})
        display(drop, out)
    except ImportError:
        pass

### 4. Exercise, Worked Answer, and Evidence Boundary

**Exercise:**  
Why is $\sum_i |\Delta_i|$ presented only as a descriptive summary rather than an adequacy theorem?

---

**Worked Answer:**  
While the absolute magnitude sum $\sum_i |\Delta_i|$ detects that errors occurred without allowing cancellation, it discards the directional identity of the failures: it cannot distinguish a catastrophic LGGR integration drop ($-1$) from a false-positive composite bypass ($+1$). Full diagnosis requires retaining the explicit pointwise records.

---

**Evidence Boundary:**  
This lesson proves that signed defect sums admit cancellation under finite aggregation. It establishes why dashboards reporting aggregate zero do not demonstrate composition soundness.